# Is `also_buy` just popularity?

`also_buy` is the field every complementary-product pair in this project is
built from, and it is not raw behaviour — it is Amazon's own recommendation
output, already ranked and truncated before we see it. This notebook asks how
much of it is explained by one thing: **being a popular item in a
complementary category**.

The question matters because of what the two-tower work found. In
`ttn/ttn_complementary.ipynb` the trained model returns near-identical top-10
lists for every query in a category — 200 different bed frames asking for
Mattresses & Box Springs drew on a pool of 41 items — and a per-category
popularity `groupby` outscores the model 0.2136 to 0.1179 on Recall@10. A
separate test there found that conditioning on the query item added nothing
over conditioning on the target category alone. If `also_buy` is essentially
popularity, all of that is explained, and the modelling effort should move to a
different target rather than to a better loss.

### How this is measured

**Popularity** is the number of distinct reviewers an item has, from
`Home_and_Kitchen_filtered.csv`. Behavioural, dense, and independent of
`also_buy` itself — unlike `also_buy` in-degree, which would be circular.

**Ranked within the item's own category**, the `cat_2 > cat_3 > cat_4` path,
parsed with the same rule the pair pipeline uses. `also_buy` lists span
categories, so a global percentile would confuse "is this a popular mattress"
with "are mattresses popular".

**Every metadata item with a non-empty `also_buy`**, not just the ones that
survived the TTN filtering, so the conclusion is about the variable rather than
about earlier filtering choices.

### What would count as an answer

Three things, in increasing strength:

1. Where `also_buy` targets sit in their own category's popularity
   distribution, against an explicit null of a random item from the same
   category.
2. A **popularity-replaceability** score: rebuild each item's `also_buy` list
   from its category mix plus popularity alone, and measure the overlap.
3. The conditional test — whether two *different* source items pointing at the
   same category receive the same items. Correlation with popularity is not
   enough on its own, because popular items genuinely are bought more; this is
   the step that separates "popularity with category routing" from "genuine
   item-to-item signal".

In [1]:
import ast
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# notebooks/ sits one level below the repo root, next to data/
ROOT = Path("..").resolve()
DATA_DIR = ROOT / "data"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from complementary_cats_pairs.categories import parse_category_levels

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 46)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# How popularity is defined here, decided before any of it was run:
#   * distinct reviewers per asin -- behavioural, dense, independent of also_buy
#   * ranked WITHIN the item's own cat_2 > cat_3 > cat_4 path, because also_buy
#     lists span categories and a global percentile would confuse "is this a
#     popular mattress" with "are mattresses popular"
#   * every metadata item carrying a non-empty also_buy, not only the ones that
#     survived the TTN filtering in ttn/ttn_complementary.ipynb
SEED = 0
rng = np.random.default_rng(SEED)

## 1. Load

Metadata carries `also_buy` and the category path; the review table supplies popularity. Both are large, so only the needed columns are read.

In [2]:
# --- Metadata: one row per asin -------------------------------------------
meta = pd.read_csv(
    DATA_DIR / "meta_Home_and_Kitchen_filtered.csv",
    usecols=["asin", "also_buy", "category", "title", "brand", "price"],
    dtype={"asin": str, "also_buy": str, "category": str},
    low_memory=False,
).drop_duplicates("asin")
print(f"metadata rows: {len(meta):,}")

# --- Reviews: the popularity signal ---------------------------------------
reviews = pd.read_csv(
    DATA_DIR / "Home_and_Kitchen_filtered.csv",
    usecols=["reviewerID", "asin"],
    dtype={"reviewerID": str, "asin": str},
    low_memory=False,
)
print(f"reviews: {len(reviews):,} rows over {reviews.asin.nunique():,} asins")

metadata rows: 1,285,392
reviews: 6,898,955 rows over 189,172 asins


## 2. Category path and popularity

`parse_category_levels` is imported from `complementary_cats_pairs.categories` rather than reimplemented, so a category is cleaned here exactly as it is in the pair pipeline. `cat_1` is `Home & Kitchen` for every row, so the node is levels 2–4.

Percentile is computed **within** the node. A node holding a single item has no meaningful percentile, so those are left null rather than scored 1.0 by construction.

In [3]:
# --- Category path, parsed with the same rule the pipeline uses ------------
levels = parse_category_levels(meta["category"], n_levels=4)
meta = pd.concat([meta.drop(columns=["category"]), levels], axis=1)
# cat_1 is "Home & Kitchen" throughout; the node is levels 2-4, as in the TTN.
meta["node"] = (meta["cat_2"].fillna("Missing").astype(str) + " > "
                + meta["cat_3"].fillna("Missing").astype(str) + " > "
                + meta["cat_4"].fillna("Missing").astype(str))

# --- Popularity: distinct reviewers per asin ------------------------------
pop = reviews.groupby("asin")["reviewerID"].nunique().rename("reviewers")
meta = meta.join(pop, on="asin")
meta["reviewers"] = meta["reviewers"].fillna(0).astype(int)

# Percentile WITHIN the node. `rank(pct=True)` puts the most-reviewed item in a
# node at 1.0; a node of one item is undefined, so those are dropped from any
# percentile statistic rather than being scored 1.0 by construction.
node_size = meta.groupby("node")["asin"].transform("size")
meta["node_size"] = node_size
meta["pop_pct"] = (meta.groupby("node")["reviewers"].rank(pct=True, method="average")
                   .where(node_size > 1))
print(f"nodes: {meta.node.nunique():,}   median node size: {meta.node_size.median():,.0f}")
print(f"items with at least one review: {(meta.reviewers > 0).mean():.1%}")
meta[["asin", "title", "node", "reviewers", "pop_pct"]].head(3)

nodes: 1,663   median node size: 7,825
items with at least one review: 14.7%


,asin,title,node,reviewers,pop_pct
0,0001487795,You Are Special Today Red Plate [With Red ...,Kitchen & Dining > Dining & Entertaining >...,0,0.443
1,0002020300,Vicks Inhaler Relief for Cold Sinus Nasal ...,Home Dcor > Candles & Holders > Candles,0,0.436
2,0006564224,Artistic Churchware Communion Cup Filler: ...,Kitchen & Dining > Dining & Entertaining >...,0,0.432


## 3. `also_buy` as edges

One row per (source, target). Targets that do not appear in this metadata cannot be given a popularity percentile — they are counted and reported before being dropped, since a silent drop here would bias every number that follows.

In [4]:
# --- also_buy -> one row per (source, target) edge -------------------------
def as_list(value):
    if not isinstance(value, str) or value in ("", "[]"):
        return []
    try:
        parsed = ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return []
    return parsed if isinstance(parsed, list) else []


has_ab = meta["also_buy"].fillna("[]").ne("[]")
print(f"items with a non-empty also_buy: {has_ab.sum():,} of {len(meta):,} ({has_ab.mean():.1%})")

sources = meta.loc[has_ab, ["asin", "also_buy"]].copy()
sources["targets"] = sources["also_buy"].apply(as_list)
sources["n_also_buy"] = sources["targets"].str.len()
print(f"also_buy list length: mean {sources.n_also_buy.mean():.1f}  "
      f"median {sources.n_also_buy.median():.0f}  max {sources.n_also_buy.max()}")

edges = (sources[["asin", "targets"]].explode("targets")
         .rename(columns={"asin": "source", "targets": "target"}).dropna())
edges = edges[edges.source != edges.target]
print(f"\nedges: {len(edges):,}")

# Attach each side's node and popularity. Targets outside this catalogue cannot
# be scored, so they are counted and then excluded -- reported, not hidden.
info = meta.set_index("asin")[["node", "reviewers", "pop_pct", "node_size", "title"]]
edges = edges.join(info.add_prefix("src_"), on="source").join(info.add_prefix("tgt_"), on="target")
unknown = edges.tgt_node.isna()
print(f"targets not present in this metadata: {unknown.sum():,} ({unknown.mean():.1%}) -- dropped")
edges = edges[~unknown & edges.tgt_pop_pct.notna()].copy()
print(f"scoreable edges: {len(edges):,} over {edges.source.nunique():,} source items")

items with a non-empty also_buy: 184,110 of 1,285,392 (14.3%)
also_buy list length: mean 20.5  median 10  max 100

edges: 3,779,613
targets not present in this metadata: 2,587,034 (68.4%) -- dropped
scoreable edges: 1,192,059 over 144,879 source items


## 4. One item at a time

Before aggregating anything: for a handful of real items, what are the popularity percentiles of the items in their `also_buy` list? The source items are picked by list length, not by outcome.

In [5]:
# --- The question, one item at a time -------------------------------------
# "If an item has 5 items in also_buy, what are those five items' popularity
# scores?" -- shown for a few real items before anything is aggregated.
def show_item(source_asin, max_rows=12):
    row = meta.loc[meta.asin == source_asin].iloc[0]
    sub = edges[edges.source == source_asin]
    print(f"SOURCE  {str(row.title)[:64]:<64}")
    print(f"        {row.node[:70]}")
    print(f"        {row.reviewers:,} reviewers, {row.pop_pct:.0%} percentile in its own node\n")
    print(f"its also_buy list ({len(sub)} scoreable of {int(row_n(source_asin))} listed):")
    print(f"  {'popularity percentile':>21}  {'reviewers':>9}  {'node size':>9}  category / title")
    for r in sub.head(max_rows).itertuples(index=False):
        print(f"  {r.tgt_pop_pct:>20.0%}  {r.tgt_reviewers:>9,}  {r.tgt_node_size:>9,.0f}  "
              f"{r.tgt_node.split(' > ')[-1][:22]:<22} {str(r.tgt_title)[:38]}")
    print(f"\n  mean percentile of this list: {sub.tgt_pop_pct.mean():.0%}"
          f"   (a random item in the same nodes would average 50%)\n")


row_n = lambda a: sources.loc[sources.asin == a, "n_also_buy"].iloc[0]

# Three source items picked by list length, not by outcome.
picks = (edges.groupby("source").size().rename("n")
         .loc[lambda s: (s >= 4) & (s <= 8)].index.to_numpy())
for a in rng.choice(picks, 3, replace=False):
    show_item(a)

SOURCE  My Table Talk&reg; Acrylic Palm Tree Classic Series Large Bowl  
        Kitchen & Dining > Dining & Entertaining > Serveware
        0 reviewers, 42% percentile in its own node

its also_buy list (7 scoreable of 11 listed):
  popularity percentile  reviewers  node size  category / title
                   87%        5.0     31,190  Glassware & Drinkware  My Table Talk&reg; Set of 4 - Acrylic 
                   42%        0.0     25,347  Serveware              My Table Talk&reg; Acrylic Palm Tree C
                   43%        0.0     31,190  Glassware & Drinkware  My Table Talk&reg; Set of 4 - Acrylic 
                   43%        0.0     31,190  Glassware & Drinkware  My Table Talk&reg; Set of 4 - Acrylic 
                   99%       37.0      8,963  Candleholders          Gifts &amp; Decor 2-Palm Tree Tropical
                   98%       34.0     18,764  Bar Tools & Drinkware  CoasterStone AS1990 Pam Absorbent Ston
                   43%        0.0     31,190  Glassware

## 5. The distribution, against an explicit null

For every edge, draw a random item from the **same node** as the real target. If `also_buy` were indifferent to popularity the two distributions would coincide; the gap between them is the effect size.

In [6]:
# --- Aggregate: where do also_buy targets sit in their own node? -----------
# The null is explicit: draw, for every edge, a random item from the SAME node
# as the real target. If also_buy ignored popularity the two would coincide.
by_node = {n: g["pop_pct"].to_numpy()
           for n, g in meta[meta.pop_pct.notna()].groupby("node")}
null_pct = np.concatenate([
    rng.choice(by_node[n], size=c, replace=True)
    for n, c in edges.tgt_node.value_counts().items() if n in by_node])

real = edges.tgt_pop_pct.to_numpy()
print(f"{'':<34}{'mean':>8}{'median':>9}{'share >0.9':>12}{'share <0.5':>12}")
for name, v in (("also_buy targets", real), ("random item in the same node", null_pct)):
    print(f"{name:<34}{v.mean():>8.3f}{np.median(v):>9.3f}"
          f"{(v > 0.9).mean():>12.1%}{(v < 0.5).mean():>12.1%}")

print(f"\nAlso for reference, the source items themselves: {edges.groupby('source').src_pop_pct.first().mean():.3f}")

                                      mean   median  share >0.9  share <0.5
also_buy targets                     0.814    0.945       61.9%       26.2%
random item in the same node         0.500    0.437        9.3%       82.4%

Also for reference, the source items themselves: 0.662


## 6. Popularity-replaceability

The headline number. Keep the category mix of each real `also_buy` list, then refill each category with its most-reviewed items — overlap with the real list is the share of `also_buy` that needs nothing but popularity to reproduce.

The control is as important as the score: the identical construction, but drawing a *random* item from each category, says how much overlap the category routing alone already produces. The score is only evidence for the hypothesis to the extent it exceeds that control.

In [7]:
# --- Headline: how much of also_buy does "popularity + category" explain? --
# For each source item, keep the category mix of its real also_buy list, then
# refill each category with its MOST-REVIEWED items. Overlap with the real list
# is the share of also_buy that needs nothing but popularity to reproduce.
#
# The control matters as much as the score: the same construction using a
# RANDOM item from each category instead of the most popular one says what
# overlap the category routing alone would have produced.
ranked = {n: g.sort_values("reviewers", ascending=False)["asin"].to_numpy()
          for n, g in meta[meta.pop_pct.notna()].groupby("node")}

N_SOURCES = 20_000          # raise to None to score every source item
grouped = list(edges.groupby("source"))
if N_SOURCES and len(grouped) > N_SOURCES:
    grouped = [grouped[i] for i in rng.choice(len(grouped), N_SOURCES, replace=False)]

hit_pop, hit_rand, sizes = [], [], []
for src, g in grouped:
    real_set = set(g.target)
    mix = g.tgt_node.value_counts()
    pop_pick, rand_pick = set(), set()
    for node, c in mix.items():
        pool = ranked.get(node)
        if pool is None:
            continue
        pool = pool[pool != src]
        pop_pick.update(pool[:c])
        rand_pick.update(rng.choice(pool, size=min(c, len(pool)), replace=False))
    hit_pop.append(len(real_set & pop_pick) / len(real_set))
    hit_rand.append(len(real_set & rand_pick) / len(real_set))
    sizes.append(len(real_set))

hit_pop, hit_rand, sizes = map(np.asarray, (hit_pop, hit_rand, sizes))
print(f"scored {len(hit_pop):,} source items, {sizes.sum():,} also_buy entries\n")
print(f"POPULARITY-REPLACEABILITY  {hit_pop.mean():.3f}")
print(f"  same construction, random item per category (control)  {hit_rand.mean():.3f}")
print(f"  share of source items fully reproduced (overlap = 1.0) {(hit_pop == 1).mean():.1%}")
print(f"  share with no overlap at all                           {(hit_pop == 0).mean():.1%}")
print("\n1.0 would mean also_buy is exactly 'the most popular items in these")
print("categories'. The control is what category routing alone already buys.")

scored 20,000 source items, 165,394 also_buy entries

POPULARITY-REPLACEABILITY  0.029
  same construction, random item per category (control)  0.002
  share of source items fully reproduced (overlap = 1.0) 0.4%
  share with no overlap at all                           83.4%

1.0 would mean also_buy is exactly 'the most popular items in these
categories'. The control is what category routing alone already buys.


## 7. Does the source item matter at all?

The discriminating test. Two different source items whose `also_buy` lists point at the same category — do they get the same items?

A high Jaccard with few distinct targets means there is effectively one list per category and the source item is decoration. A low Jaccard means genuine item-to-item signal that a model could learn. This is the result that determines whether retargeting on review co-occurrence is worth doing.

In [8]:
# --- The discriminating test: does the SOURCE item matter at all? ----------
# Popularity correlation alone cannot settle the question, because popular items
# genuinely are bought more. This asks the conditional version: two different
# source items whose also_buy lists point at the SAME category -- do they get
# the same items? If also_buy were item-specific the overlap would be low.
pairs_by_node = (edges.groupby("tgt_node")["source"].nunique()
                 .sort_values(ascending=False))
rows = []
for node in pairs_by_node.head(15).index:
    sub = edges[edges.tgt_node == node]
    srcs = sub.source.unique()
    if len(srcs) < 20:
        continue
    take = rng.choice(srcs, size=min(60, len(srcs)), replace=False)
    lists = [set(sub.loc[sub.source == s, "target"]) for s in take]
    lists = [l for l in lists if l]
    jac = [len(a & b) / len(a | b) for i, a in enumerate(lists) for b in lists[i + 1:]]
    rows.append({"node": node.split(" > ")[-1][:26], "source_items": len(take),
                 "items_in_node": int(meta[meta.node == node].shape[0]),
                 "mean_jaccard": float(np.mean(jac)),
                 "distinct_targets": len(set().union(*lists)),
                 "total_slots": sum(len(l) for l in lists)})
overlap = pd.DataFrame(rows)
print(overlap.to_string(index=False))
print(f"\nmean Jaccard across different source items pointing at the same category: "
      f"{overlap.mean_jaccard.mean():.3f}")
print("High Jaccard + few distinct targets = one list per category, and the")
print("source item is decoration. Low Jaccard = genuine item-to-item signal.")

                      node  source_items  items_in_node  mean_jaccard  distinct_targets  total_slots
    Decorative Accessories            60          35700         0.001               255          284
                 Serveware            60          25347         0.003               116          125
                   Missing            60         101862         0.001               221          227
                   Missing            60           7139         0.004                72           85
                Dinnerware            60          36409         0.002               148          167
Baking Tools & Accessories            60          11299         0.002               202          234
     Collectible Figurines            60          38463         0.003               277          335
     Glassware & Drinkware            60          31190         0.002               186          199
                 Ornaments            60          37885         0.001               301    

## 8. Verdict

*Fill in once the cells above have run — the numbers should be read together:*

- **Section 5** says whether `also_buy` targets are drawn from the popular end of their category, and by how much against the null.
- **Section 6** says how much of `also_buy` can be reconstructed from popularity plus category routing alone, net of the random control.
- **Section 7** says whether anything is left that depends on *which* item you started from.

If 6 is high and 7 shows high overlap with few distinct targets, the hypothesis holds: `also_buy` is popularity routed through complementary categories, the two-tower model's identical per-category lists are the correct answer to the question it was asked, and the way forward is a different training target — review co-occurrence — rather than a different loss.